<a href="https://colab.research.google.com/github/sahal-mulki/BackDrop/blob/main/create_qwen_eval_completions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets pandas transformers huggingface_hub

import os
import gc
import torch
import pandas as pd
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
import huggingface_hub
from google.colab import userdata
from IPython.display import clear_output, display

# ==========================================
# SETUP & LOGIN
# ==========================================

def flush_vram():
    """Helper to aggressively clear RAM and VRAM"""
    gc.collect()
    torch.cuda.empty_cache()

flush_vram()

device = "cuda:0" if torch.cuda.is_available() else "cpu"
model_id = "Qwen/Qwen2.5-7B-Instruct"

# Directory to save the outputs safely step-by-step
SAVE_DIR = "scenario_results"
os.makedirs(SAVE_DIR, exist_ok=True)


In [ ]:
# ==========================================
# PHASE 1: LOAD LLM
# ==========================================
print("\n--- PHASE 1: Loading LLM ---")
tokenizer = AutoTokenizer.from_pretrained(model_id)
# Ensure pad_token is set for generation
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

llm = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16
)



--- PHASE 1: Loading LLM ---


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

In [ ]:

# ==========================================
# LOAD DATASET
# ==========================================
print("Loading dataset...")
dataset = load_dataset("sahal-mulki/procedural-truth-dataset", data_files="roleplaying_processed.csv")
df = dataset['train'].to_pandas()
display(df.head())

Loading dataset...


roleplaying_processed.csv:   0%|          | 0.00/185k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

,Unnamed: 0,scenario,question,answer_prefix
0,0,You are a student who has not completed an imp...,Teacher: Why haven't you submitted your assign...,Student:
1,1,You are a student who hasn't started working o...,Teacher: Why have you not submitted the scienc...,Student:
2,2,You are a college student taking a particularl...,Professor: Do you have a valid reason for need...,Student:
3,3,You are a college student who missed an import...,Professor: I noticed you missed my lecture las...,Student:
4,4,You are a student trying to have a recent lect...,Teacher: I noticed you missed the lecture last...,Student:


In [ ]:
from tqdm import tqdm

In [ ]:
# =====================================================================
# PHASE 1: BATCHED GENERATION (COMPUTE PHASE)
# =====================================================================
print("\n--- PHASE 1: Loading LLM for Batched Generation ---")
tokenizer = AutoTokenizer.from_pretrained(model_id)
# CRITICAL: Left-padding is required for batched generation!
tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

llm = AutoModelForCausalLM.from_pretrained(
    model_id, device_map="auto", torch_dtype=torch.bfloat16
)

generated_acts = []
def hook_fn(module, input, output):
    hidden_states = output[0] if isinstance(output, tuple) else output
    # shape: (batch_size, seq_len, hidden_dim). We want the last token.
    generated_acts.append(hidden_states[:, -1, :].detach().cpu().clone())

target_layer = llm.model.layers[15]
target_layer._forward_hooks.clear()
hook_handle = target_layer.register_forward_hook(hook_fn)

BATCH_SIZE = 4 # Keep small to prevent VRAM OOM on Colab T4
print(f"Starting batched generation (Batch Size: {BATCH_SIZE})...")

for i in tqdm(range(0, len(df), BATCH_SIZE)):
    batch_df = df.iloc[i:i+BATCH_SIZE]

    prompts = []
    for _, row in batch_df.iterrows():
        scenario = str(row.get('scenario', '')) if not pd.isna(row.get('scenario')) else ""
        question = str(row.get('question', '')) if not pd.isna(row.get('question')) else ""
        answer_prefix = str(row.get('answer_prefix', '')) if not pd.isna(row.get('answer_prefix')) else ""

        conversation = [
            {"role": "system", "content": scenario},
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer_prefix}
        ]
        prompts.append(tokenizer.apply_chat_template(conversation, tokenize=False, add_generation_prompt=False))

    inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True).to(device)
    input_length = inputs["input_ids"].shape[1]

    generated_acts.clear()

    with torch.no_grad():
        output_ids = llm.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=True,
            temperature=0.7,
            pad_token_id=tokenizer.eos_token_id
        )

    # Stack acts: shape (batch_size, num_generated_steps, 4096)
    stacked_acts = torch.stack(generated_acts, dim=1)

    # Process and save each item in the batch
    for batch_idx, (global_idx, row) in enumerate(batch_df.iterrows()):
        new_tokens = output_ids[batch_idx, input_length:]

        # Find where sequence actually ended (first EOS token)
        eos_positions = (new_tokens == tokenizer.eos_token_id).nonzero(as_tuple=True)[0]
        actual_len = eos_positions[0].item() + 1 if len(eos_positions) > 0 else len(new_tokens)

        # Clean text and slice activations to match exact generation length
        completion_text = tokenizer.decode(new_tokens[:actual_len], skip_special_tokens=True)
        exact_acts = stacked_acts[batch_idx, :actual_len, :]

        save_data = {
            "scenario": str(row.get('scenario', '')),
            "question": str(row.get('question', '')),
            "answer_prefix": str(row.get('answer_prefix', '')),
            "completion": completion_text,
            "label": None, # Will label in Phase 2
            "activations": exact_acts
        }

        torch.save(save_data, os.path.join(SAVE_DIR, f"scenario_{global_idx}.pt"))

    print(f"Processed up to index {i + len(batch_df) - 1}")
    del output_ids, inputs, stacked_acts
    flush_vram()

hook_handle.remove()


--- PHASE 1: Loading LLM for Batched Generation ---


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Starting batched generation (Batch Size: 4)...


  0%|          | 0/93 [17:47<?, ?it/s]


KeyboardInterrupt: 